In [ ]:
from datasets import load_dataset

ds = load_dataset("Orange/lc_quad2-sparqltotext")
print(ds)          # shows train/validation/test
print(ds["train"][0])

In [1]:
# train.py
#
# Create dataset/train.json, dataset/validation.json, dataset/test.json
# from Orange/lc_quad2-sparqltotext with 4 fields:
#  1) question          (natural language)
#  2) sparql_dbpedia18  (DBpedia SPARQL)
#  3) sparql_wikidata   (Wikidata SPARQL)
#  4) answer            (solved_answer as textual result)
#
# Filters out examples with empty solved_answer ([]).

import os
import json
from datasets import load_dataset


def convert_split(split, name, out_dir):
    records = []

    kept = 0
    skipped_missing_fields = 0
    skipped_empty_answer = 0

    for ex in split:
        question = ex.get("question")
        sparql_dbpedia = ex.get("sparql_dbpedia18")
        sparql_wikidata = ex.get("sparql_wikidata")
        solved_answer = ex.get("solved_answer")

        # Require required fields
        if not question or not sparql_dbpedia or not sparql_wikidata:
            skipped_missing_fields += 1
            continue

        # Require a non-empty result set
        # (solved_answer is usually a list; skip [] / None)
        if not solved_answer:
            skipped_empty_answer += 1
            continue

        records.append(
            {
                "question": question.strip(),
                "sparql_dbpedia18": sparql_dbpedia.strip(),
                "sparql_wikidata": sparql_wikidata.strip(),
                "answer": solved_answer,  # keep as list
            }
        )
        kept += 1

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{name}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print(
        f"[{name}] kept={kept} | skipped_missing_fields={skipped_missing_fields} | skipped_empty_answer={skipped_empty_answer}"
    )
    print(f"Saved: {out_path}")


def main():
    print("Loading Orange/lc_quad2-sparqltotext ...")
    ds = load_dataset("Orange/lc_quad2-sparqltotext")
    print(ds)

    out_dir = "dataset"
    convert_split(ds["train"], "train", out_dir)
    convert_split(ds["validation"], "validation", out_dir)
    convert_split(ds["test"], "test", out_dir)


if __name__ == "__main__":
    main()

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Orange/lc_quad2-sparqltotext ...
DatasetDict({
    train: Dataset({
        features: ['uid', 'NNQT_question', 'paraphrased_question', 'question', 'simplified_query', 'sparql_dbpedia18', 'sparql_wikidata', 'answer', 'solved_answer', 'subgraph', 'template', 'template_id', 'template_index'],
        num_rows: 21101
    })
    validation: Dataset({
        features: ['uid', 'NNQT_question', 'paraphrased_question', 'question', 'simplified_query', 'sparql_dbpedia18', 'sparql_wikidata', 'answer', 'solved_answer', 'subgraph', 'template', 'template_id', 'template_index'],
        num_rows: 3010
    })
    test: Dataset({
        features: ['uid', 'NNQT_question', 'paraphrased_question', 'question', 'simplified_query', 'sparql_dbpedia18', 'sparql_wikidata', 'answer', 'solved_answer', 'subgraph', 'template', 'template_id', 'template_index'],
        num_rows: 6024
    })
})
[train] kept=10551 | skipped_missing_fields=0 | skipped_empty_answer=10550
Saved: dataset/train.json
[validation] k

In [3]:
import os
import json
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATA_DIR = "dataset"
OUTPUT_DIR = "./qwen25-coder-sparql2text"
MAX_SEQ_LEN = 512
USE_WIKIDATA_SPARQL = False

# -------------------------
# Tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# 4-bit QLoRA loading
# -------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# -------------------------
# LoRA
# -------------------------
peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, peft_cfg)
model.print_trainable_parameters()

# -------------------------
# Dataset
# -------------------------
data_files = {
    "train": os.path.join(DATA_DIR, "train.json"),
    "validation": os.path.join(DATA_DIR, "validation.json"),
}

raw = load_dataset("json", data_files=data_files)

def to_text(ex):
    sparql = ex["sparql_wikidata"] if USE_WIKIDATA_SPARQL else ex["sparql_dbpedia18"]
    question = ex["question"]

    text = f"""You are an expert in knowledge graphs.

Convert the SPARQL query into a clear natural language question.

### SPARQL
{sparql}

### Question
{question}{tokenizer.eos_token}
"""
    return {"text": text}

raw = raw.map(to_text)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )

train_ds = raw["train"].map(
    tokenize_fn,
    batched=True,
    remove_columns=raw["train"].column_names,
)

val_ds = raw["validation"].map(
    tokenize_fn,
    batched=True,
    remove_columns=raw["validation"].column_names,
)

# -------------------------
# Data collator
# -------------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# -------------------------
# Training args
# -------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,

    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,
    report_to="none",
)

# -------------------------
# Trainer
# -------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# -------------------------
# Train
# -------------------------
trainer.train()

# -------------------------
# Save best model + tokenizer
# -------------------------
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Optional: save logs
with open(os.path.join(OUTPUT_DIR, "log_history.json"), "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

print("Saved best model to", OUTPUT_DIR)

Loading weights: 100%|██████████| 339/339 [00:05<00:00, 66.26it/s, Materializing param=model.norm.weight]                               


trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


Map: 100%|██████████| 1526/1526 [00:00<00:00, 5836.83 examples/s]
/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
500,0.289846,0.292370
1000,0.273695,0.277329
1500,0.256015,0.269726
2000,0.259063,0.262923
2500,0.250131,0.257688
3000,0.231196,0.255848
3500,0.232391,0.253225


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more 

Saved best model to ./qwen25-coder-sparql2text
